# Passo 7: dados de 2026 no app

O Banco Central publicou o SCR.data de **jan a set/2026**. Este notebook acrescenta esses meses
ao app **sem treinar o modelo de novo**: o `app/modelo.joblib` do passo 6 continua o mesmo.

O que muda:
1. Os **mesmos 386 recortes** ganham os meses de 2026.
2. O alvo é recalculado com a mesma regra (+0,5 p.p. **e** +20% em 6 meses). Agora já sabemos o que
   aconteceu com os meses-base de **jul/2025 a mar/2026**.
3. Como o modelo foi treinado só com meses-base até **jun/2025**, esses meses são um **teste de
   verdade**: previsões feitas para um período que o modelo nunca viu.

In [ ]:
import joblib
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score

from src.dados import CHAVES_RECORTE, PASTA_PROCESSED, montar_painel
from src.variaveis import VARIAVEIS, VARIAVEIS_CAT, VARIAVEIS_NUM, criar_variaveis

# Mesma regra do alvo do passo 3
HORIZONTE = 6
ALTA_MIN_PP = 0.5
ALTA_MIN_REL = 0.20

# Último mês-base usado no treino do modelo final (passo 6)
FIM_TREINO = pd.Timestamp("2025-06-30")

## 1. Painel de 2026 para os mesmos 386 recortes

`montar_painel([2026])` usa a mesma função do passo 2 (baixa o ZIP, se faltar, e soma por recorte).
Os anos de 2022 a 2025 vêm do `painel_filtrado.parquet`, exatamente como o modelo os viu.

In [ ]:
painel_antigo = pd.read_parquet(PASTA_PROCESSED / "painel_filtrado.parquet")
painel_2026 = montar_painel([2026])

# Fica só com os 386 recortes do estudo (merge "inner" com a lista de recortes)
recortes = painel_antigo[CHAVES_RECORTE].drop_duplicates()
for coluna in CHAVES_RECORTE:  # texto, para o merge não depender das categorias de cada ano
    recortes[coluna] = recortes[coluna].astype(str)
    painel_2026[coluna] = painel_2026[coluna].astype(str)
painel_2026 = painel_2026.merge(recortes, on=CHAVES_RECORTE, how="inner")

print(f"Meses de 2026: {painel_2026['data_base'].dt.strftime('%m/%Y').unique().tolist()}")
print(f"Recortes com dados em 2026: {painel_2026.groupby(CHAVES_RECORTE).ngroups} de {len(recortes)}")
print("Meses por recorte em 2026:", painel_2026.groupby(CHAVES_RECORTE).size().value_counts().to_dict())

## 2. Juntar os anos e completar o calendário

O cálculo do alvo e das variáveis usa `shift`, que anda **linhas**, não meses. Se faltar um mês num
recorte, o `shift(6)` pegaria o mês errado. Por isso, cada recorte recebe **todos os meses** do
período; um mês que o Banco Central não publicou fica com valores vazios (`NaN`).
Nada é inventado: as contas que dependem desse mês também ficam vazias.

In [ ]:
painel_antigo = painel_antigo.copy()
for coluna in CHAVES_RECORTE:
    painel_antigo[coluna] = painel_antigo[coluna].astype(str)
painel = pd.concat([painel_antigo, painel_2026], ignore_index=True)

# Grade completa: cada recorte × cada mês do período
meses = pd.date_range(painel["data_base"].min(), painel["data_base"].max(), freq="ME")
grade = recortes.merge(pd.DataFrame({"data_base": meses}), how="cross")
painel = grade.merge(painel, on=CHAVES_RECORTE + ["data_base"], how="left")

faltando = painel[painel["carteira_ativa"].isna()]
print(f"Período: {meses.min():%m/%Y} a {meses.max():%m/%Y} ({len(meses)} meses)")
print(f"Linhas: {len(painel)} ({len(recortes)} recortes x {len(meses)} meses)")
print(f"Meses sem dados no BC: {len(faltando)}")
print(faltando[CHAVES_RECORTE + ["data_base"]].to_string(index=False))

for coluna in CHAVES_RECORTE:
    painel[coluna] = painel[coluna].astype("category")
painel = painel.sort_values(CHAVES_RECORTE + ["data_base"]).reset_index(drop=True)

## 3. Alvo e variáveis (mesmas regras dos passos 3 e 4)

In [ ]:
painel = criar_variaveis(painel)  # calcula taxa_inad e as demais variáveis

taxa_futura = painel.groupby(CHAVES_RECORTE, observed=True)["taxa_inad"].shift(-HORIZONTE)
subiu_pp = (taxa_futura - painel["taxa_inad"]) > ALTA_MIN_PP
subiu_rel = taxa_futura > painel["taxa_inad"] * (1 + ALTA_MIN_REL)
painel["alvo"] = (subiu_pp & subiu_rel).astype("Int8")
painel.loc[taxa_futura.isna() | painel["taxa_inad"].isna(), "alvo"] = pd.NA

com_alvo = painel.dropna(subset=["alvo"])
print(f"Último mês-base com alvo: {com_alvo['data_base'].max():%m/%Y}")
print("Alvo = 1 (%) por semestre do mês-base:")
semestre = com_alvo["data_base"].dt.year.astype(str) + "-S" + ((com_alvo["data_base"].dt.month > 6) + 1).astype(str)
print((100 * com_alvo.groupby(semestre)["alvo"].mean()).round(1).to_string())

Conferência: até dez/2025 as variáveis precisam ser **idênticas** às do app atual (passo 6).
O alvo dos meses-base de jul a dez/2025 era vazio e agora é conhecido, então ele fica fora da comparação.

In [ ]:
colunas = ["data_base"] + VARIAVEIS_CAT + ["carteira_ativa"] + VARIAVEIS_NUM + ["alvo"]
app_antigo = pd.read_parquet("app/dados_app.parquet")
ate_2025 = painel[painel["data_base"] <= app_antigo["data_base"].max()]
pd.testing.assert_frame_equal(
    ate_2025[colunas[:-1]].reset_index(drop=True).astype({c: str for c in VARIAVEIS_CAT}),
    app_antigo[colunas[:-1]].reset_index(drop=True).astype({c: str for c in VARIAVEIS_CAT}),
)
mesmo_alvo = app_antigo["alvo"].notna()
assert (ate_2025["alvo"].reset_index(drop=True)[mesmo_alvo] == app_antigo["alvo"][mesmo_alvo]).all()
print("Até dez/2025: variáveis e alvos idênticos aos do app atual.")

## 4. Teste de verdade: jul/2025 a mar/2026

O modelo do app (`modelo.joblib`, **sem retreino**) dá a nota de cada recorte nesses meses-base.
Como o futuro de 6 meses já está nos dados, dá para conferir com as mesmas métricas do passo 5.

In [ ]:
modelo = joblib.load("app/modelo.joblib")

novos = painel[(painel["data_base"] > FIM_TREINO)].dropna(subset=VARIAVEIS_NUM + ["alvo"]).copy()
novos["nota_risco"] = modelo.predict_proba(novos[VARIAVEIS])[:, 1]
y = novos["alvo"].astype(int)

print(f"Meses-base: {novos['data_base'].min():%m/%Y} a {novos['data_base'].max():%m/%Y}, {len(novos)} linhas")
print(f"Positivos: {100 * y.mean():.1f}% (AP de um chute)")
print(f"AUC: {roc_auc_score(y, novos['nota_risco']):.3f}")
print(f"AP:  {average_precision_score(y, novos['nota_risco']):.3f}")

# Regra simples do passo 5 como referência: ordenar pelo atraso de 15 a 90 dias
print(f"Regra simples (atraso 15-90): AUC {roc_auc_score(y, novos['taxa_atraso_15_90']):.3f}, "
      f"AP {average_precision_score(y, novos['taxa_atraso_15_90']):.3f}")

In [ ]:
# Mês a mês: dos 20 recortes de maior nota, quantos tiveram alta relevante de fato?
def top20(grupo):
    return pd.Series({
        "recortes": len(grupo),
        "altas (%)": round(100 * grupo["alvo"].astype(int).mean(), 1),
        "altas no top 20": int(grupo.nlargest(20, "nota_risco")["alvo"].astype(int).sum()),
    })

novos.groupby(novos["data_base"].dt.strftime("%Y-%m")).apply(top20, include_groups=False)

## 5. Arquivo do app

O `dados_app.parquet` passa a ir até set/2026. O `modelo.joblib` **não muda**.

In [ ]:
painel[colunas].to_parquet("app/dados_app.parquet", index=False)
print(f"dados_app.parquet: {len(painel)} linhas, {painel['data_base'].min():%m/%Y} a "
      f"{painel['data_base'].max():%m/%Y}")